# P05 on PolyBench with a T4 GPU (Phase 3.2)

Runs on **Colab** or **Kaggle** (it works in `/content` or `/kaggle/working`, whichever exists).
P05's choices were made for the host in `machine/kaggle-t4.json` (4 threads): run it
there. The second cell checks the thread count and says so if it differs.

- **Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Keep the
  tab open; if the session disconnects, reconnect and Run all again: the run resumes from the
  rows already in the CSV (unless the runtime was reset).
- **Kaggle:** File → Import Notebook (this file). In the right-hand panel, Session options:
  **Accelerator → GPU T4 x2** and **Internet on** (both need a phone-verified account). Then
  **Save Version → Save & Run All (Commit)**: it runs in the background (up to 12 hours) with
  the browser closed, and the CSV appears under the version's **Output**.

Installing the NVIDIA HPC SDK takes about 10 minutes; the benchmark run takes longer (30
kernels, 6 variants, 3 runs each at the LARGE size, plus two strict-IEEE builds of ludcmp and lu).

Variants per kernel: `seq` (nvc -O3, no OpenMP), `seq-mp` (the same source with the OpenMP
flags: what `-mp=gpu` alone does), `p05` (P05's own choices with the `kaggle-t4`
machine model: GPU, host threads or sequential per loop), `p05-naive` (every loop P05 can
offload goes to the GPU), and both again as `-warm` (the device started before `main`, so
the timed kernel doesn't pay the ~0.16 s start-up). For ludcmp and lu, `seq-ieee` and
`p05-ieee` are built with `-Kieee -Mnofma` (correctly rounded division, no fused
multiply-adds), checked against each other.
Correctness: exact hex dumps against `seq`. `gpu_launches` counts kernel launches (0 = every guard chose the host).

**When it finishes, download `polybench-t4.csv` (Colab: the last cell offers it; Kaggle: the
version's Output) and give it to Claude**; it goes to `eval/polybench/results/`.

Generated by `eval/polybench/make_gpu_notebook.py` at P05 964cc94 (30
kernels, LARGE_DATASET, 73 GPU pragmas in the gated rewrites): edit that and `gpu_run.py`, not this
notebook.


In [ ]:
# Where to work: Colab's /content or Kaggle's /kaggle/working.
import base64, gzip, os
ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.makedirs(ROOT, exist_ok=True)
os.environ['P05_ROOT'] = ROOT  # the %%bash cells cd here
os.chdir(ROOT)
RUNNER = gzip.decompress(base64.b64decode("""
H4sIAAAAAAAC/51aaXPbSJL9zl9RA8eGgW0SomS3d4ZudoRGR7djfShkuSc6NAoMCBREjEAAjQIl
chX67/syq3ARoKxpfRBxVGVl5fHyKLz6y8FaFQeLOD2Q6b3It+UyS9+MLMu6WPpKijfu0UxcTH8U
WSousmT7d5kGy4MT8dY9cg/FQ1wuhS9+ufgm7JMs8Rfi6u1YpPeBmKzy+W2+dtzRSODP0BV45BXr
1M23YjLJFyIHyQWRxK0qAlHIhyIupRLXkwnGKfHmhi4DdU9TXfzejEZfZRJNgiwt/TiVobBVGSbx
AhwmW2cm5L2fHNR0D1b+nfRo2TQr5SLL7mhtuVrIUIm4HI/8NKT9vVb12mMRp5pKQFtqaHnlWzfO
t+liLFScBlKUSykqsqPAT1+XIkiyVL8oZJ6puMyKrWs2t8wSLLq72MoPJUkXc0ahvJdJlq9kWuJ5
sMT+tIwn1Z351axNwNC/VZbOIGMscHDrlzI8+OlOFqlMfnYDCF6vlj2kIlhmcQDJ2tDWWJxAZeWy
kD44ygqh5B9rLBr7iWNopX58L7u0JpBEHGwFvyJBy2Irvh6fn4kky3JIU0AGIouiJPNDsyWyjdHo
QhZCUxL2x+PLX8680+Or469nV2Mx+fLGYf7lH6L+o4lZEd/GqZ/AnjLxJZfppws9DKY1MEzLiZ7o
sSJK/FvsFpPzwr9d+Qq28bD0y8o0R6L757PmwgwiKjMmtMwUtpRBPWQkcakE7ayM01vYbZyWoJDD
M+o/Fn+l2XHFh5GHHjxh2dFgfWEGm5cPfrEaN8P4fpdNYkz5K9DXzgdTXeXlVpR+cStLELyNIXn4
jljIKCvIYmDOtr9LB/6jymIdwEDJ3mkpN4CIiH4Zr7ARozAIEJK5hQZzf6vYTHdIwWphWTAzBSbK
yTrHcotsXYqpe/hOqMoSCBo+f7k6+yqOpkfvJofTyeE7Z7xL7GEZB4QptCZBQEri9vMclgfRgxRz
kcH9jDXEUkotNLraJUdmRdojRS3WcVIaf+LBE7IRmMh9MBOT/6UnYvIpzaKVP+6LqyhkUCZbUWTr
NIR4wvg+VsQQkYeQorXC09U6KeM82U78MFTOmEFplxbUUjGgZaze1/yLWMFTZXAHWv4tVAcTrHbp
iiuZJKrP2iqPE+ina50QGrQhoiJbQZzw9AS+EsqCBgRLHxrdJWQn6zCAc4VxFMlCicVWHLlv5eRv
5OopC460ev7puKvIw8nRX53Rr+QtBlJm4sunC+/zt0/e1a+XZ8enX8UcO67wRoeKmTgS94AhRTHi
RIs3lQpzfXHx5ePvfz/7fPKrd/oNhI4vL49//8r6A65EQgIExb1fxD62mWNHvGu58YNytJQbLQhA
K0nGLyBJ1jnECBu1LR5nEeqRVSbkN4ocJ4Hw7oHBvHvgvXQgcONt7FABNA8QoGhnfCPx14gLBKqf
fzvBlj98PLv0jk9OPIjnw/nv70cUCk1cmoqV9BHRNGx23BVsxZHtEEQrWUMPxHKZPSiBHZADSLY6
A00nX38TPjvjVkRxGqvle1Yysanu4hwBhqcmJPHtCOMKOInKMCiMFXw/hbRBTgdtBbEzaki1XmE3
DzSc8FyVGRYOXcoHRvEqz2BQYBxCVbK6R0yuLjNVXRX1a7Vcl3FS360XeZEhDNUj1VaNRucfzj6y
kVxbWrLWWFhGwXRpLIcugTHlmq8Ipbz6ii+Ml1pt97VW/saDdj1ZFDSGcoFKbzyZNeGZIMFUMKJ1
G0okGol1M3oF0F6xGNdFUAP8mLGTUK8FqRhUFnFQig9nZ2facEe/HV9+OP58xdu0LVgjEafwhd9z
P1Gy+wPw4FEIV62BV8W6N67jx7YFKOF9UjR68QwdcWhedTE0z4ylDfeXaE3Yv0I9t7vMc3Mr+Nsr
Lppc81aNHNw+j7wZ/eP48pN3Aj3ArF8hkwPuSfETwMJd/jzyPL+E7hbrUnqebbeU6jgU4co4EPdZ
zCHFo93YdOeIR7BNsLsB3en70SttQQJUK2df+bldZgTIM7GhPGsjfpiLw/ejJ/avEdK/iIDGDlbh
WCAdn39GSjLmcAzg5TvOlYSQWCSEfdmZcjEwLrLU0S/cdR5i6zZl84C4xyf9vCy2s1quBWY3nujW
SwZ+Xq4L6WGxHOtp0ZVyU10SSy1+zK/T0JWYDhhx9QWlTmPcIT/HMHMFL9SMbgKZtwHBvdLkzjZ5
DMye7VKdHEKrlbdjXCUwA/J2IaOxuM1KIyFI9DUj/evxPqgXDdQjWSofpNSJSrpGeVAol5RCpGJo
BQvN50S+x5eJJ2NkO1N+54/FAgLGFFchbSlth/mqbiqSiUxt3xF/mfPVwukTNnEYpDmg2VacRpae
/5AV2Mi8XpNSis1YbCmV+784t4mHFkUstyH+WzZgckBEznWTCHSshOmCaLTFQsyBS7aLAGtvnPHO
k21jBUazv/nJWp4VRVZ0SfbW6K/T0G/TfQHtF0rPiCTakPSjHXaaFxsyHHA1MOg/WKitLoQim6/H
wl8oGytMQNoRB/xGP3LMuy0uDuXkzXTqaGJmPdsqM2R7PqzWImY17Z/mNBiZGnCuYQkk+DVcJUh8
pQSX73or5Dueh9yhBMwplNRjCu2qZTX00PXBNj3vPs3JwoE9uV8uXbBLvzYNwxunOxQM3O0bTO+a
4RhC1XoYF8qupwJ1NrEqveyOQWiXOPC3RfzfSHzbUy1d1rS0kSFm2/VUGmE5Ltdgto4KOwsA/AoS
wfVN/fwVxPbh89XF1aV39fvFmedxdoaUiJxP19u3a78IkdX/cnLC5cFJ4lNaDliTEVXxMZK7ihzg
byGf2wMP6G+CH3c2YMXcNsB0E5Fqk8ELx97h2pm+F0//TLsuQaHgmlUTBGNRLTHJLHMjfhCWS+B7
41xPb8gzpl3H6ArNmpzurDqnug65VG2CpgLiHbdsjzN23mdfLvkCPFFWGZexzuG4PbPyizsvga1Y
CNRIjqMuZ0YW14lLqVluO9dHsxsGzoQUFzEmV+9aDC6ReUpjkWNxR7yMda3RYheh4hhwmm+pPKEY
orcFUzDTaQl6zt4AC5eUUGyFzYRmdRFTE2zKGOe9STh1JWCqPZSckLIpB2DHKZUVtAi11erIxRop
gkHrIilWT8EQpbY2785pLCJ8zi6XYTEBmwxCvA0DPnhuNaQXvpJMG3w4HVtLs7IeFSuwYIdOV2Vt
QAi7iEp6i2i/GEM6p+m0Qh+mCc+RI4WKTMq23KU1MIgDE222b3EFHCFyHJcqENsZnFkJYJhuRbsE
iTzxA2m/tv5r6h4lkbCQnODGx4Wz87b7cpBwn9mQWO1gQj87CxvT5rrEGLYubYx586ZhaLq2MeY+
FpRbz3WSPnuZjSzidMgW9tjdcCgI94YAuekBZ6h3ApyyrUm4Ru20a55Wa5lX4l91/m/dwTT+BQDz
Cy7pudHGjmcaqI3XRmALTmtcsUXN1xjQcs3a/x9QylIl3eRe+XZQchXedJAGerUeaWNPk8dBz3pq
4bguuV1aQZsvXbWcWkbKgLMFOeko76JoAL2y1/gC+p8OtmN6gm2Pu/rw6cxqQqbuxc6FjUW/vLFu
SD1mXcjQ49dVfkxkqWhiotc3TsUiB5W+HzSUyDqZVJsS99QqUo0MUO4wO2ZqEBBLmssftIB+IAF9
gPE+pxQescvTdwOWozXyn0+kZKA6jKCE4GbUF8Z1ndzccHZICVJLlNhUwvU3x3Q40E0LIGAqHm2Y
qhddE3aTg2Ag4BtU0ZUquK4iqFZCgmxHQRNA62tL9+8in8IXJRCTw5tdcAJDVOU1EIWS0wAUbKKF
OsTjo7XTZbRm1Hap7ME0jZyn9h5Apcu/vGYqV3QqceV9OT//+OX4FBY6rxyj6d93OTXlp2kU2Yxg
mjRVzFUs4cft2EFvqOArpItUMPSTxC6sgWZBKf0VaZyIGcMNvjcVdTAeyoSiY3eqYRoYHhi+OU80
DPu5TvK5r+ceF7drOoG6oDsEZakCaJR673PPC7PA8zq6RR7oGCquH4aeb6YDf5GKgI1C/rGmit40
EZYyyedW/yyxhldrLzmA2T56DThz5qhPwnRrng+y9hOlSAWqkIm/Tsq5xfd7Rwfqvj24dkc6iqN3
+ycG7XnpfbB/KFTJJxOqPcOcWokJ/s2D4H9+fGa+ttj27NOz8+NvH6/6YGWSTRJh3w04y2odqZmO
rNq/tIki7aU75337ZzYt3nKbyznS4YbG0d5ZdEY8OOXN/oVM+2ho1rvptDIoJYMMSaPIkVZjlf2M
mwLGosQDhdPc+u/9Y2sBvmRwfUL1rNZMn3nn7AeJcXN6aZvt7R5yCWtvviosUj2sdKyDxySKcj5y
LxD45jAwVCX62E8vcSdlrviIFJnS+FnC+oAvVoIrYdwl0pyc6uOng2RtzlXoVK2S7vNSGtBBo9Rr
SxOmqJesrQqtCs5IQJFRj2iqpjnHJXCdUBC3FOCayLHzfk7ZZke2nB31MjVTWSM40kmpqUgaLBj+
M9nVrgbMHGrEMJAyccdETo7GqXzgMMYdSALLfie4KbM1a+p+qHQmchRv7aI+n7kBDl/XRzRILche
uAAFDfcUJnnJKZMdteIvn3szU1xEtPvA5zCDz1l5ToerO4293Fe6+0SHj/Mut1Cpb/FOKRrNrao9
inEVH/+gOqiwuQOtz5lqLWNaSxS6YtKZnm0ESbvinI92tnCrZkUrEWn3tjgsIujaPcWbAgexsp2M
8C4MTY5Vd1wUY6n2m+/0bLk1LR6fqP4wYKATrcnP5siTyqBRu3DuHmO16jx9Ap2K6qiq1xnlAX1O
2RMG2R1kme3W5zyzOd3pZut8Pra7Oq9VYagJ7ivZ4aT/FikrrfVCtrCKfadrSIeIsun/SVq689Kt
0Bec02tzoMW0Eqg3r8+2RvsAoEOFt8qlXa3EPRbXBZYSmg4o2auyViodu+l89lAdLWl9ziENI9S5
thsTqefMRHWm3y7aDsf70Kx71jrX7NSHrfMg7zLzSrS+C2At0Nl7dXjrdv2CPppAtCqKcXMsP+cT
o6k75aKie5CAnSQZQffC1X2Q73dATMbJbRD654x6XaAB26CTOFqFShkqYoY6VL2ypDX52hr8uIBr
FOuwbzG7Zdw1VXnm8C69b47vtP52z/CqP87uOXx0sv3+uEbWar2yD5uOqqZguqpNuDOx/eTb6bGJ
7pbTJ0uYRfv7Z2ppk08GCFtExKr8PxmU7GDR2jEaWqZKCs0MuOPkUBtzZMlNXIrHInjqi1omFEUI
Geb7kMFA9DXZC2msA8YdOhX4YvQz3LKJU3hrjj017TGTdr6LZi8Erz/nIOacfb+HkK4GvcSEX+CP
zeiDi7HQH3vMdedgYjoHlNnRFxlzcHc9O5pObwbsB8p0o2StlgMmy812O7Ie72aH754E99Vmhz8+
iXaDYiYemf7h0fTmic/2QGyn//isAPmjlGoLfIJE3y/cWb02tsdKpw+yNK5TTTPQotZoP+zXDbi8
1L2/5xk123/WNTjGIzbc9d6wYFz9RZOtj0xV1m0b2dwc2lHrcxaif8bVZ3pz7SrPJNVCtD4LmsMW
8DNz39w+8T41Lph+8fNk2p8TzWswbIfD71HQ3zERDyvgHEvHmbnvIs0K37+cEJxFGMQEwY2hw58H
8BEUU3+ejPEui5ev7GDOtqv5aJp81xO432xXUft8b6/fPepVZn99GqwcO5LZkYm2n9epn752ZodT
9230pIR4ZAXO/raPXq2wx+pqyMVpH0GSKUnVwCim43Ji2fNYHJ5HLTTPs7QD6X7a6P8BQSikzFAv
AAA=
""")).decode()
open(os.path.join(ROOT, 'gpu_run.py'), 'w').write(RUNNER)
print('working in', ROOT)


In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
nproc; grep -m1 'model name' /proc/cpuinfo
if [ "$(nproc)" != "4" ]; then
  echo "WARNING: this host has $(nproc) threads, but P05's rewrites were made for kaggle-t4 (4)."
  echo "Its CPU choices may not fit this host: say so when you send the CSV."
fi


In [ ]:
%%bash
# NVIDIA HPC SDK (nvc -mp=gpu): the compiler the GPU evaluation uses (NOTES.md, 2026-10-02).
set -e
if ls /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc >/dev/null 2>&1; then echo "already installed"; exit 0; fi
curl -s https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK \
  | gpg --dearmor --yes -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' \
  > /etc/apt/sources.list.d/nvhpc.list
apt-get update -qq
PKG=$(apt-cache search --names-only '^nvhpc-[0-9]+-[0-9]+$' | awk '{print $1}' | sort -V | tail -1)
echo "Installing $PKG"; apt-get install -y -qq $PKG > /tmp/nvhpc.log 2>&1 || { tail -20 /tmp/nvhpc.log; exit 1; }
ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc


In [ ]:
%%bash
# PolyBench/C 4.2.1 at the commit eval/llm_baseline/fetch_polybench.sh pins.
set -e
cd "$P05_ROOT"
[ -d polybench/.git ] || git clone -q https://github.com/MatthiasJReisinger/PolyBenchC-4.2.1 polybench
git -C polybench checkout -q 3e872547cef7e5c9909422ef1e6af03cf4e56072
test "$(git -C polybench rev-parse HEAD)" = 3e872547cef7e5c9909422ef1e6af03cf4e56072 && echo "polybench at 3e872547cef7e5c9909422ef1e6af03cf4e56072"


In [ ]:
# P05's rewrites (made with -machine machine/kaggle-t4.json), packed by
# make_gpu_notebook.py (gzip + base64: one long line freezes the editor).
import base64, gzip, json, os
ROOT = os.environ['P05_ROOT']
PACKED = gzip.decompress(base64.b64decode("""
...
MFIYKYwURlpyRvr6PyTi2hpsWAwA
""")).decode()
FILES = json.loads(PACKED)
for p, text in FILES.items():
    os.makedirs(os.path.dirname(os.path.join(ROOT, 'rewrites', p)), exist_ok=True)
    open(os.path.join(ROOT, 'rewrites', p), 'w').write(text)
print(len(FILES), 'files')


In [ ]:
%%bash
cd "$P05_ROOT"
NVC=$(ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc | sort -V | tail -1)
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
$NVC --version | grep -v '^$' | head -1
python3 gpu_run.py --pb polybench --src rewrites --cc $NVC --omp-flags "-mp=gpu -gpu=cc$CC" \
  --threads $(nproc) --runs 3 --dataset LARGE_DATASET --csv polybench-t4.csv


In [ ]:
import csv, math, os
CSV = os.path.join(os.environ['P05_ROOT'], 'polybench-t4.csv')
rows = list(csv.DictReader(open(CSV)))
t = {(r['kernel'], r['variant']): float(r['time_s']) for r in rows if r['status'] == 'ok' and r['time_s']}
bad = [(r['kernel'], r['variant'], r['status'], r['correct']) for r in rows
       if r['status'] != 'ok' or r['correct'] not in ('', 'exact', 'tolerance')]
print('not ok / not correct:', bad or 'none')
for v in ('seq-mp', 'p05', 'p05-naive', 'p05-warm', 'p05-naive-warm', 'p05-ieee'):
    s = 'seq-ieee' if v == 'p05-ieee' else 'seq'
    xs = [t[(k, s)] / t[(k, v)] for (k, vv) in t if vv == v and (k, s) in t]
    if xs:
        print(f"{v:15} geomean speedup over {s} {math.exp(sum(map(math.log, xs)) / len(xs)):.2f} ({len(xs)} kernels)")
try:
    from google.colab import files
    files.download(CSV)
except ImportError:  # Kaggle: files in /kaggle/working are the version's Output
    from IPython.display import FileLink, display
    print('CSV at', CSV)
    display(FileLink(os.path.relpath(CSV)))
